# Ten-player next-frame position model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joyalzzy/playable-replays/blob/ml/ten-player-position-ai.ipynb)

This Google Colab-only notebook downloads a revision-pinned batch from [`maknee/league-of-legends-decoded-replay-packets`](https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets), builds fixed-time frames through the dataset publisher's companion Gym, and trains a separate QLoRA adapter.

Each model input contains **exactly ten heroes** with dataset-native `x`/`z` positions. Each target contains possible positions for the **same ten player IDs at the next complete fixed-time frame**. Frames missing any hero position are excluded. Player names are never exported. Whole matches—not individual frames—are held out for testing.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.request import Request, urlopen

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if not IN_COLAB:
    raise RuntimeError("Data preparation and training for this model are restricted to Google Colab")

import torch
CUDA_AVAILABLE = torch.cuda.is_available()

HF_DATASET_REPO = "maknee/league-of-legends-decoded-replay-packets"
HF_DATASET_REVISION = "04f9c7350e9ffcc689b731875ad9baf3ff6eaa6d"
HF_REPLAY_FILE = "13_2/batch_001.jsonl.gz"
HF_REPLAY_SHA256 = "c4b8846d8b088481afee92ad9d660fc959532438e64b9616b09a1690caf7e6e7"
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # @param {type:"string"}
MAX_GAMES = 8  # @param {type:"integer"}
FRAME_SECONDS = 1.0  # @param {type:"number"}
MAX_GAME_SECONDS = 2400.0  # @param {type:"number"}
MAX_FRAME_PAIRS_PER_GAME = 2000  # @param {type:"integer"}
EVAL_MATCH_FRACTION = 0.20  # @param {type:"number"}
MAX_SEQ_LENGTH = 4096  # @param {type:"integer"}
MAX_STEPS = 100  # @param {type:"integer"}
SEED = 3407  # @param {type:"integer"}
RUN_TRAINING = False  # @param {type:"boolean"}
DOWNLOAD_ARTIFACT = False  # @param {type:"boolean"}
OUTPUT_DIR = Path("/content/playable-replays-output/ten-player-position-model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if min(MAX_GAMES, FRAME_SECONDS, MAX_GAME_SECONDS, MAX_FRAME_PAIRS_PER_GAME, MAX_SEQ_LENGTH, MAX_STEPS) <= 0:
    raise ValueError("All limits must be positive")
if not 0 < EVAL_MATCH_FRACTION < 1:
    raise ValueError("EVAL_MATCH_FRACTION must be in (0, 1)")
print(json.dumps({"python": platform.python_version(), "inColab": IN_COLAB, "cuda": CUDA_AVAILABLE, "outputDir": str(OUTPUT_DIR)}, indent=2))

## Install the publisher's Gym and training dependencies

The Gym is the dataset publisher's recommended interface and exposes time-stepped `game_state.heroes`, `game_state.positions`, and `get_position(net_id)`. This avoids treating a movement command's final waypoint as if it were an engine-exact current position.

In [ ]:
packages = ["league-of-legends-decoded-replay-packets-gym", "huggingface_hub"]
if RUN_TRAINING:
    if not CUDA_AVAILABLE:
        raise RuntimeError("QLoRA training requires a Colab CUDA runtime")
    packages.extend(["unsloth", "datasets", "trl"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages])
print("Colab dependencies installed.")

In [ ]:
POSITION_PIPELINE_URL = "https://raw.githubusercontent.com/joyalzzy/playable-replays/ml/position_pipeline.py"
if not Path("position_pipeline.py").is_file():
    request = Request(POSITION_PIPELINE_URL, headers={"User-Agent": "playable-replays-colab-bootstrap/1.0"})
    with urlopen(request, timeout=30) as response:
        module_source = response.read(1_000_001)
    if len(module_source) > 1_000_000:
        raise ValueError("position_pipeline.py exceeded the bootstrap size cap")
    Path("position_pipeline.py").write_bytes(module_source)

from position_pipeline import (
    build_training_example,
    grouped_split,
    record_order_key,
    score_position_prediction,
    summarize_results,
    validate_frame_pair,
    write_jsonl,
)
import league_of_legends_decoded_replay_packets_gym as lol_gym
from huggingface_hub import hf_hub_download

replay_batch_path = Path(hf_hub_download(repo_id=HF_DATASET_REPO, filename=HF_REPLAY_FILE, repo_type="dataset", revision=HF_DATASET_REVISION))
digest = hashlib.sha256()
with replay_batch_path.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(chunk)
actual_sha256 = digest.hexdigest()
if actual_sha256 != HF_REPLAY_SHA256:
    raise ValueError(f"Replay batch checksum mismatch: {actual_sha256}")
print(f"Downloaded and verified in Colab: {replay_batch_path} ({actual_sha256})")

## Build complete ten-player frame pairs

A frame is accepted only when the Gym reports exactly ten heroes and a position for every hero ID. The next target is the immediately following accepted fixed-time frame, provided its time delta matches `FRAME_SECONDS`. Hero dictionaries are reduced to stable network IDs and champion identifiers; account/player names are omitted.

In [ ]:
def complete_frame(game_state: Any) -> list[dict[str, Any]] | None:
    heroes = game_state.heroes
    if not isinstance(heroes, dict) or len(heroes) != 10:
        return None
    players = []
    for net_id, hero in sorted(heroes.items(), key=lambda item: str(item[0])):
        position = game_state.get_position(net_id)
        if position is None:
            return None
        champion = hero.get("champion") if isinstance(hero, dict) else None
        if not isinstance(champion, str) or not champion:
            champion = "unknown-champion"
        players.append({"playerId": str(net_id), "champion": champion[:80], "position": {"x": float(position.x), "z": float(position.z)}})
    return players

dataset = lol_gym.ReplayDataset([str(replay_batch_path)])
dataset.load(max_games=MAX_GAMES)
env = lol_gym.LeagueReplaysEnv(dataset, max_time=MAX_GAME_SECONDS, time_step=FRAME_SECONDS)
observation, info = env.reset()
frame_pairs: list[dict[str, Any]] = []
completed_match_ids: set[str] = set()
current_match_id: str | None = None
previous_frame: dict[str, Any] | None = None
complete_frame_index = -1
pairs_in_current_match = 0
max_steps = MAX_GAMES * (math.ceil(MAX_GAME_SECONDS / FRAME_SECONDS) + 10)

for _ in range(max_steps):
    observed_match_id = str(info.get("game_id", f"game-{len(completed_match_ids)}"))
    if observed_match_id != current_match_id:
        current_match_id = observed_match_id
        previous_frame = None
        complete_frame_index = -1
        pairs_in_current_match = 0
    game_state = info["game_state"]
    players = complete_frame(game_state)
    if players is not None:
        complete_frame_index += 1
        current_frame = {"index": complete_frame_index, "time": float(game_state.current_time), "players": players}
        if previous_frame is not None and pairs_in_current_match < MAX_FRAME_PAIRS_PER_GAME:
            time_delta = current_frame["time"] - previous_frame["time"]
            if math.isclose(time_delta, FRAME_SECONDS, rel_tol=0, abs_tol=1e-6):
                frame_pairs.append({
                    "matchId": current_match_id,
                    "frameIndex": previous_frame["index"],
                    "nextFrameIndex": current_frame["index"],
                    "frameTimeSeconds": previous_frame["time"],
                    "nextFrameTimeSeconds": current_frame["time"],
                    "players": previous_frame["players"],
                    "nextPlayers": [{"playerId": player["playerId"], "position": player["position"]} for player in current_frame["players"]],
                    "metadata": {
                        "datasetRepo": HF_DATASET_REPO, "datasetRevision": HF_DATASET_REVISION, "datasetFile": HF_REPLAY_FILE, "datasetFileSha256": HF_REPLAY_SHA256,
                        "coordinateMethod": "publisher Gym game_state dataset-native x/z positions at fixed-time frames",
                    },
                })
                pairs_in_current_match += 1
        previous_frame = current_frame
    observation, reward, terminated, truncated, next_info = env.step(0)
    if terminated or truncated:
        completed_match_ids.add(current_match_id)
        if len(completed_match_ids) >= min(MAX_GAMES, len(dataset)):
            break
        observation, info = env.reset()
        previous_frame = None
    else:
        info = next_info
env.close()

records = [validate_frame_pair(record, index) for index, record in enumerate(frame_pairs)]
records.sort(key=record_order_key)
positions = [record_order_key(record) for record in records]
if len(positions) != len(set(positions)):
    raise ValueError("Duplicate (matchId, frameIndex) records are not allowed")
if len({record['matchId'] for record in records}) < 2:
    raise ValueError("At least two games with complete ten-player frames are required")
write_jsonl(OUTPUT_DIR / "frame-pairs.jsonl", records)
print(json.dumps({"loadedGames": len(dataset), "completedGames": len(completed_match_ids), "completeFramePairs": len(records)}, indent=2))

In [ ]:
train_records, eval_records = grouped_split(records, eval_fraction=EVAL_MATCH_FRACTION, seed=SEED)
train_match_ids = {record["matchId"] for record in train_records}
eval_match_ids = {record["matchId"] for record in eval_records}
if train_match_ids & eval_match_ids:
    raise AssertionError("Held-out match IDs overlap training")
train_examples = [build_training_example(record) for record in train_records]
eval_examples = [build_training_example(record) for record in eval_records]
write_jsonl(OUTPUT_DIR / "train.jsonl", train_examples)
write_jsonl(OUTPUT_DIR / "eval.jsonl", eval_examples)
split_report = {"trainMatches": len(train_match_ids), "evalMatches": len(eval_match_ids), "trainFrames": len(train_examples), "unusedEvalFrames": len(eval_examples), "overlap": sorted(train_match_ids & eval_match_ids)}
(OUTPUT_DIR / "split-report.json").write_text(json.dumps(split_report, indent=2), encoding="utf-8")
print(json.dumps(split_report, indent=2))
preview = train_examples[0]
print("\nINPUT:")
print(json.dumps(json.loads(preview["messages"][1]["content"]), indent=2)[:8000])
print("\nNEXT FRAME TARGET:")
print(json.dumps(json.loads(preview["messages"][2]["content"]), indent=2)[:8000])

## Train the separate position adapter

Set `RUN_TRAINING=True` and rerun the notebook on a Colab GPU. Training and test data remain separated by complete match IDs.

In [ ]:
model = tokenizer = trainer = None
training_metrics = None
if RUN_TRAINING:
    from datasets import Dataset
    from transformers import TrainingArguments
    from trl import SFTTrainer
    from unsloth import FastLanguageModel, is_bfloat16_supported

    model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=SEED, use_rslora=False,
    )
    def add_text(batch: dict[str, list[Any]]) -> dict[str, list[str]]:
        return {"text": [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) for messages in batch["messages"]]}
    train_dataset = Dataset.from_list(train_examples).map(add_text, batched=True)
    eval_dataset = Dataset.from_list(eval_examples).map(add_text, batched=True)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_dataset, eval_dataset=eval_dataset, dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=1, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=min(5, max(1, MAX_STEPS // 10)), max_steps=MAX_STEPS, learning_rate=2e-4,
            fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), logging_steps=1, eval_strategy="steps", eval_steps=max(1, MAX_STEPS // 5),
            optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear", seed=SEED, output_dir=str(OUTPUT_DIR / "checkpoints"), report_to="none",
        ),
    )
    training_result = trainer.train()
    training_metrics = training_result.metrics
    print(json.dumps(training_metrics, indent=2, default=str))
else:
    print("Training skipped. Set RUN_TRAINING=True in a Colab GPU runtime.")

## Test on unused matches

The model must return exactly ten known player IDs. The held-out report measures JSON validity, complete-frame validity, and Euclidean position error in dataset-native coordinates.

In [ ]:
heldout_results: list[dict[str, Any]] = []
heldout_report = None
if RUN_TRAINING:
    FastLanguageModel.for_inference(model)
    for example in eval_examples:
        input_ids = tokenizer.apply_chat_template(example["messages"][:2], add_generation_prompt=True, tokenize=True, return_tensors="pt").to(model.device)
        outputs = model.generate(input_ids=input_ids, attention_mask=torch.ones_like(input_ids), max_new_tokens=768, do_sample=False, use_cache=True)
        generated_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()
        try:
            prediction = json.loads(generated_text)
        except json.JSONDecodeError:
            prediction = generated_text
        expected = json.loads(example["messages"][2]["content"])
        scored = score_position_prediction(prediction, expected)
        scored.update({"matchId": example["metadata"]["matchId"], "frameIndex": example["metadata"]["frameIndex"], "rawOutput": prediction})
        heldout_results.append(scored)
    write_jsonl(OUTPUT_DIR / "heldout-predictions.jsonl", heldout_results)
    heldout_report = summarize_results(heldout_results)
    (OUTPUT_DIR / "heldout-report.json").write_text(json.dumps(heldout_report, indent=2), encoding="utf-8")
    print(json.dumps(heldout_report, indent=2))
else:
    print("Held-out generation runs after training; match-disjoint split artifacts are ready.")

In [ ]:
if RUN_TRAINING:
    adapter_dir = OUTPUT_DIR / "ten-player-position-lora"
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    manifest = {
        "createdAt": datetime.now(timezone.utc).isoformat(), "baseModel": BASE_MODEL, "method": "QLoRA ten-player next-frame position prediction",
        "datasetRepo": HF_DATASET_REPO, "datasetRevision": HF_DATASET_REVISION, "datasetFile": HF_REPLAY_FILE, "datasetFileSha256": HF_REPLAY_SHA256,
        "frameSeconds": FRAME_SECONDS, "trainMatches": len(train_match_ids), "evalMatches": len(eval_match_ids), "trainExamples": len(train_examples), "evalExamples": len(eval_examples),
        "inputContract": "exactly ten player IDs, champions, and dataset-native x/z positions", "outputContract": "same ten player IDs with possible next-frame x/z positions",
        "trainingMetrics": training_metrics, "heldoutEvaluation": heldout_report,
    }
    (adapter_dir / "training-manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    archive = shutil.make_archive(str(OUTPUT_DIR / "ten-player-position-lora"), "zip", root_dir=adapter_dir)
    print(f"Saved adapter: {archive}")
    if DOWNLOAD_ARTIFACT:
        from google.colab import files
        files.download(archive)
else:
    print(f"Prepared frame and split artifacts: {OUTPUT_DIR}")

## Interpretation boundary

The target is the publisher Gym's reconstructed next fixed-time position frame. It is suitable for offline sequence prediction, but it is not proof of player intent and is not guaranteed to reproduce proprietary engine state exactly. Do not mix this adapter with the packet-prediction or critical-objective movement adapters.